In [63]:
import pandas as pd
import pingouin as pg
from scipy.stats import false_discovery_control

import warnings 
warnings.filterwarnings('ignore')

In [64]:
df_main = pd.read_csv('cvd_risk_covars.csv', index_col=0)
#print(df_main.head())
#df_microbiome = pd.read_csv('microbes.csv', index_col=0)

#df_microbiome = pd.read_csv('microbes.csv', index_col=0)
#df_microbiome = pd.read_csv('microbes_filtered_20percent.csv', index_col=0)
#df_microbiome = pd.read_csv('microbes_filtered_10percent.csv', index_col=0)
#df_microbiome = pd.read_csv('microbes_filtered_5percent.csv', index_col=0)
df_microbiome = pd.read_csv('microbes_filtered_2percent.csv', index_col=0)
#df_microbiome = pd.read_csv('microbes_filtered_1percent.csv', index_col=0)
#df_microbiome = pd.read_csv('microbes_filtered_0.5percent.csv', index_col=0)

#df_microbiome = df_microbiome.iloc[:, -50:]
df_microbiome = df_microbiome.apply(pd.to_numeric, errors='coerce')
print(df_microbiome.dtypes.value_counts())
df_microbiome.head()

int64    567
Name: count, dtype: int64


,Peredibacter,Legionella,Lineage_IIa,Brevibacillus,Lineage_IIb,Hydrogenedensaceae,SM2D12,WD2101_soil_group,Eubacterium_oxidoreducens_group,SAR324_clade.Marine_group_B.,...,Subdoligranulum,Escherichia.Shigella,Faecalibacterium,Agathobacter,Blautia,Collinsella,Intestinibacter,Clostridium_sensu_stricto_1,Bifidobacterium,Romboutsia
LS6651,0,0,0,0,0,0,0,0,0,0,...,3769,2920,5327,1519,6516,4055,1072,10954,4498,9112
LS8700,0,0,0,0,0,0,0,0,0,0,...,7529,1845,6716,4483,8035,6316,8560,26816,4974,13406
LS8710,0,0,0,0,0,0,0,0,0,0,...,9829,1764,10090,3152,14819,6179,5324,5997,6572,8469
LS8729,0,0,0,2,0,0,0,0,0,0,...,5318,586,3853,2214,7852,25859,3082,4280,1778,9901
LS8751,0,0,0,0,0,0,0,0,0,0,...,4522,2116,6216,8549,6205,3879,4541,5892,8605,7616


In [65]:
df_main['setting'] = df_main['setting'].map({'Rural':0, 'Urban':1})
df_main['sex'] = df_main['sex'].map({'male':0, 'female':1})
df_main['infection'] = df_main['infection'].map({'Negative':0, 'Positive':1})

df_main.head()

,slabno,setting,age,sex,bmi,occpcat2,exc4,smok,alcoh,infection,insulin,BP_Dia,Chol_T,Chol_L,glucose,BP_Sys
LS6651,LS6651,0,31,0,26.55186,Category 2,Not active,No,0,1,3.72,84.0,4.99,3.24,4.52,121.0
LS8700,LS8700,0,42,0,20.91135,Category 3,Not active,No,Yes,1,8.22,72.5,4.19,2.53,5.60,103.0
LS8710,LS8710,0,19,0,22.52859,Category 2,Very active,No,0,1,4.41,70.5,3.41,2.20,4.00,120.0
LS8729,LS8729,0,60,0,21.35779,Category 2,A bit active,No,Yes,0,2.93,71.5,4.93,3.12,4.65,119.0
LS8751,LS8751,0,32,0,23.42209,Category 2,Not active,No,0,1,3.44,60.5,3.46,2.19,4.46,120.0


In [66]:
# Define your CVD outcomes and covariates
cvd_outcomes = ['insulin', 'BP_Dia', 'Chol_T', 'Chol_L', 'glucose', 'BP_Sys']
covariates = ['age', 'sex', 'setting']

# Get microbe columns (or specify a subset)
microbe_columns = df_microbiome.columns.tolist()

In [67]:
df = pd.concat([df_main, df_microbiome], axis=1)
#print(df.columns)
print(df.dtypes)
df.tail()
print(df['infection'].value_counts())

slabno                          object
setting                          int64
age                              int64
sex                              int64
bmi                            float64
                                ...   
Collinsella                      int64
Intestinibacter                  int64
Clostridium_sensu_stricto_1      int64
Bifidobacterium                  int64
Romboutsia                       int64
Length: 583, dtype: object
infection
1    128
0     81
Name: count, dtype: int64


In [68]:
covariates = ['age', 'sex', 'setting']
microbe_cols = microbe_columns
cvd_vars = cvd_outcomes

results = pd.DataFrame()

#'insulin', 'BP_Dia', 'Chol_T', 'Chol_L', 'glucose', 'BP_Sys'

cvd_vars = ['glucose']

for cvd_outcome in cvd_vars:
    print(f"Processing {cvd_outcome}...")
    
    for i, microbe in enumerate(microbe_cols):
            
            # Run mediation
            med = pg.mediation_analysis(
                data=df,
                x='infection',  # infected/uninfected (binary: 0/1)
                m=microbe,      # mediator
                y=cvd_outcome,
                covar=covariates
            )
            med['Microbe'] = microbe
            med['cvd_outcome'] = cvd_outcome

            results = pd.concat([results, med])


Processing glucose...


In [69]:
results

,path,coef,se,pval,CI[2.5%],CI[97.5%],sig,Microbe,cvd_outcome
0,Peredibacter ~ X,7.373258e-02,0.081440,0.366345,-0.086840,0.234306,No,Peredibacter,glucose
1,Y ~ Peredibacter,3.595433e-02,0.076213,0.637601,-0.114312,0.186220,No,Peredibacter,glucose
2,Total,-2.490674e-02,0.088860,0.779538,-0.200109,0.150295,No,Peredibacter,glucose
3,Direct,-2.766847e-02,0.089205,0.756752,-0.203556,0.148219,No,Peredibacter,glucose
4,Indirect,2.761732e-03,0.007844,0.864000,-0.007234,0.028517,No,Peredibacter,glucose
...,...,...,...,...,...,...,...,...,...
0,Romboutsia ~ X,2.570569e+03,1044.890171,0.014720,510.399838,4630.738007,Yes,Romboutsia,glucose
1,Y ~ Romboutsia,-4.656202e-07,0.000006,0.936841,-0.000012,0.000011,No,Romboutsia,glucose
2,Total,-2.490674e-02,0.088860,0.779538,-0.200109,0.150295,No,Romboutsia,glucose
3,Direct,-2.441325e-02,0.090390,0.787368,-0.202638,0.153811,No,Romboutsia,glucose


In [70]:
# Apply FDR correction
results['Indirect_pval_fdr'] = false_discovery_control(results['pval'])

In [71]:
results_indirect = results[results['path']== 'Indirect']
results_indirect = results_indirect[results_indirect['sig']== 'Yes']
results_indirect

,path,coef,se,pval,CI[2.5%],CI[97.5%],sig,Microbe,cvd_outcome,Indirect_pval_fdr
4,Indirect,-0.010419,0.006474,0.012,-0.029868,-0.002168,Yes,Gallicola,glucose,0.925258
4,Indirect,-0.018138,0.010920,0.044,-0.046452,-0.002300,Yes,Phoenicibacter,glucose,0.925258
4,Indirect,-0.019890,0.011557,0.024,-0.045363,-0.002453,Yes,Treponema,glucose,0.925258
4,Indirect,-0.025106,0.013256,0.032,-0.059453,-0.005971,Yes,Methanobrevibacter,glucose,0.925258


In [72]:
results_indirect.shape

(4, 10)

In [73]:
results_indirect.to_csv('glucose_helminth_mediation.csv', index=False)